# Graph Neural Networks (GNNs)
Graphs represent molecules, social networks, knowledge bases, road maps, and more. GNNs extend deep learning to graph-structured data through learnable message passing over nodes and edges.

## Why Graphs?
Many real-world systems are fundamentally relational:
- **Molecules**: atoms (nodes) + chemical bonds (edges)
- **Social Networks**: users (nodes) + friendships (edges)
- **Citation Graphs**: papers (nodes) + citations (edges)
- **Knowledge Graphs**: entities (nodes) + typed relations (edges)

Standard CNNs/MLPs cannot handle graphs because:
- Variable and irregular neighborhood sizes
- No fixed spatial grid or coordinate system
- Permutation invariance — node ordering should not matter

## Core Paradigm: Message Passing
All GNN variants follow this 3-step framework per layer:
1. **Aggregate**: collect messages from neighbors N(v)
2. **Update**: combine current state h_v with aggregated messages
3. **Readout**: pool node embeddings for graph-level predictions

```
h_v^(k) = UPDATE( h_v^(k-1),  AGG({h_u^(k-1) : u in N(v)}) )
```

## 1. Graph Convolutional Network (GCN)
Kipf & Welling (2017). Spectral-based approach using a first-order Chebyshev polynomial approximation.

Layer-wise propagation rule:
```
H^(k) = sigma( D_tilde^{-1/2} A_tilde D_tilde^{-1/2}  H^{(k-1)}  W^(k) )
```
- A_tilde = A + I (adjacency + self-loops)
- D_tilde = diagonal degree matrix of A_tilde
- W^(k) = learnable weight matrix

The symmetric normalization prevents high-degree nodes from dominating aggregation.

## 2. Graph Attention Network (GAT)
Veličković et al. (2018). Assigns **learnable attention weights** to each neighbor:

```
alpha_ij = softmax( LeakyReLU( a^T [W*h_i || W*h_j] ) )
h_i_new = sigma( sum_j alpha_ij * W * h_j )
```
Multi-head attention further improves stability — same motivation as in Transformers.

In [1]:
import numpy as np

def simple_gcn_layer(A, H, W):
    # A: adjacency matrix (N x N), H: node features (N x d_in), W: weights (d_in x d_out)
    A_hat = A + np.eye(A.shape[0])          # add self-loops
    D_inv_sqrt = np.diag(1.0 / np.sqrt(A_hat.sum(axis=1)))
    A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt  # symmetric normalization
    return np.tanh(A_norm @ H @ W)

N = 4
A = np.array([[0,1,1,0],[1,0,1,0],[1,1,0,1],[0,0,1,0]], dtype=float)
H = np.random.randn(N, 3)   # 3-dimensional node features
W = np.random.randn(3, 2)   # project to 2-dimensional
H_new = simple_gcn_layer(A, H, W)
print("Input node feature shape:", H.shape)
print("Output node feature shape:", H_new.shape)

Input node feature shape: (4, 3)
Output node feature shape: (4, 2)


## 3. GraphSAGE (Graph Sample and Aggregate)
Hamilton et al. (2017). Designed for **inductive** settings — generalizes to unseen nodes and graphs.

**Key steps per layer:**
1. Sample a fixed-size set of neighbors (not all)
2. Aggregate sampled neighbor features (mean, max, or LSTM)
3. Concatenate self-embedding with aggregated: `h_v = Concat(h_v, AGG(N_v))`
4. Apply linear layer + normalization

Scalable to graphs with billions of nodes (deployed at Pinterest with 2B+ nodes).

## 4. Message Passing Neural Networks (MPNN)
Gilmer et al. (2017). A unified framework that subsumes GCN, GAT, GraphSAGE, and most others.

**Message Phase:**
`m_v^{t+1} = SUM_{w in N(v)} M_t(h_v^t, h_w^t, e_vw)`

**Update Phase:**
`h_v^{t+1} = U_t(h_v^t, m_v^{t+1})`

**Readout (graph-level):**
`y_hat = R({h_v^T | v in G})`

Where M_t is the message function, U_t is the update function, R is a permutation-invariant readout.

## 5. Applications Overview
| Task | Example | Common GNN |
|---|---|---|
| Node classification | Predict research paper topic | GCN, GAT |
| Link prediction | Predict user friendship | GraphSAGE |
| Graph classification | Predict molecule toxicity | GIN, MPNN |
| Graph generation | Drug discovery | GraphVAE |

# Conclusions and Key Takeaways
- GNNs extend deep learning to non-Euclidean graph domains using the message passing paradigm.
- **GCN** provides a simple, strong spectral-based baseline.
- **GAT** improves with adaptive, data-dependent neighbor attention.
- **GraphSAGE** adds inductive generalization and scalability via neighborhood sampling.
- **MPNN** is the general framework that most state-of-the-art GNNs are instances of.

# Pros and Cons
**Pros:**
- Naturally handles relational structure, variable neighborhood sizes, and graph topology
- Captures both node attributes and structural information simultaneously
- Inductive methods generalize to entirely unseen nodes and graphs at inference

**Cons:**
- Over-smoothing: many GNN layers blur node representations toward identical global values
- Scalability: dense or huge graphs make full neighborhood computation intractable
- Standard GNNs cannot distinguish certain non-isomorphic graphs (bounded by 1-WL test)

# 15 Interview Questions and Answers

1. **What is the core operation in a GNN?**
   *Answer*: Message Passing — each node aggregates feature messages from its neighbors then updates its own representation based on its current state and the aggregated messages.

2. **Why can't standard CNNs/MLPs process graph data?**
   *Answer*: They require fixed-size, ordered inputs (like image grids). Graphs have variable-degree nodes, no inherent ordering, and irregular connectivity patterns.

3. **What is symmetric normalization in GCN and why is it needed?**
   *Answer*: The term D^{-1/2} A_hat D^{-1/2} normalizes so high-degree nodes don't disproportionately influence their neighbors. Without it, nodes with many connections would dominate all aggregations.

4. **What is the key advantage of GAT over GCN?**
   *Answer*: GAT learns per-edge attention weights alpha_ij, allowing the model to assign higher importance to relevant neighbors rather than weighting all neighbors equally.

5. **What is the difference between transductive and inductive GNNs?**
   *Answer*: Transductive (GCN): trained on the complete graph including test nodes (but not their labels); cannot handle new unseen nodes. Inductive (GraphSAGE): learns a function that can generate embeddings for completely new nodes never seen during training.

6. **Why is GraphSAGE preferable at large scale?**
   *Answer*: It samples a fixed-size neighborhood per node, making memory and computation predictable regardless of the total number of nodes or edges in the graph.

7. **What is over-smoothing?**
   *Answer*: After many message-passing layers, all nodes aggregate information from the entire graph and their representations converge to similar values, losing the discriminative local structural information.

8. **What is graph pooling?**
   *Answer*: Aggregating all node embeddings into a single fixed-size graph-level representation for graph classification. Simple methods: sum/mean pooling. Learned methods: DiffPool (differentiable hierarchical pooling).

9. **What is the 1-Weisfeiler-Leman (WL) test?**
   *Answer*: A classical graph isomorphism algorithm. Standard GNNs using mean/sum aggregation have been proven to be at most as expressive as 1-WL — they cannot distinguish all non-isomorphic graph pairs.

10. **How does GIN (Graph Isomorphism Network) improve upon GCN?**
    *Answer*: GIN uses a learnable epsilon parameter and a powerful MLP aggregator: h_v = MLP( (1+epsilon)*h_v + SUM_{u in N(v)} h_u ), proven to be as expressive as the 1-WL test and strictly more powerful than mean-aggregation GCN.

11. **How are edge features incorporated in GNNs?**
    *Answer*: Edge features e_vw are passed as additional input to the message function M_t(h_v, h_w, e_vw). In molecular GNNs, bond type, bond order, and aromaticity are common edge features.

12. **What is a heterogeneous graph?**
    *Answer*: A graph with multiple node types (e.g., User, Item) and/or multiple edge types (e.g., purchased, viewed). Heterogeneous GNNs (HAN, HGT) maintain type-specific parameters to handle this complexity.

13. **How are GNNs used in recommendation systems?**
    *Answer*: User-item interactions form a bipartite graph. GNNs propagate collaborative filtering signals across the graph structure to produce high-quality user and item embeddings for ranking.

14. **Name two large-scale real-world GNN deployments.**
    *Answer*: Pinterest PinSage (GraphSAGE variant for visual recommendation at billions of nodes) and Google Maps (GNN on road networks for real-time ETA prediction).

15. **What is a spectral vs spatial GNN?**
    *Answer*: Spectral GNNs (GCN) operate in the graph Fourier domain using the graph Laplacian's eigenvectors. Spatial GNNs (GraphSAGE, GAT) directly aggregate features from local neighborhoods in the node domain. Spatial methods are more scalable and interpretable.
